# Benchmark Models

diffusion 모델(`1_modeling_RV`, `2_abligation_study`) 대비 GAN 기반 벤치마크를 동일한 데이터·scaling·평가 파이프라인으로 학습/생성/평가한다. 구조는 ablation study 와 동일: 섹션별 학습+검증 → 마지막에 생성+평가+종합 테이블.

| 모델 | 설명 | 출처 |
|------|------|------|
| QuantGAN | TCN 기반 GAN (BCE + weight clipping). 원본 그대로 import | `modules/QuantGAN` |
| TailGAN (core) | MLP G/D + VaR·ES tail-score + NeuralSort. 6자산용 코어 적응 | `modules/TailGAN` |

- 두 모델 모두 **scaled 6차원 로그수익률**(LambertW 포함 `scaling`)을 학습/생성하고 `inverse_scaling` 으로 6차원 수익률 복원 → ablation 의 `w/o RV` 와 동일 전처리라 diffusion 결과와 직접 비교 가능.
- 산출물(가중치/합성데이터/CSV)은 `tag`(`quantgan`, `tailgan`)로 분리 저장.

In [ ]:
import os
import sys
import glob
import yaml
import ml_collections
import pandas as pd
import torch.nn as nn
from tqdm import tqdm
from utils.utils import *
from utils.metric import *
from utils.eval_portfolio.summary import *
from utils.eval_portfolio.strategies import log_return_to_price
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath
from torch.utils.data import DataLoader, TensorDataset

# QuantGAN 네트워크 import (modules/QuantGAN 를 namespace 패키지로 sys.path 추가)
_QG = os.path.abspath("modules/QuantGAN")
if _QG not in sys.path:
    sys.path.insert(0, _QG)
from src.baselines.networks.generators import UserGenerator
from src.baselines.networks.discriminators import UserDiscriminator

file_manager = quantDiffusionPath()
gc.collect()

# Preprocessing
원본 6차원 로그수익률을 train/test 분할하고 `scaling`(SS1→LambertW→SS2)으로 표준화한다.

In [ ]:
# --- [0. 준비] ---  벤치마크 GAN 은 원본 6차원 로그수익률을 생성 (RV 분해 없음)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 256

df = pd.read_csv("./data/indices2.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)

df_log_returns = np.log1p(df.pct_change()).dropna()
log_returns = df_log_returns.values
init_price_raw = np.array(df)[:-1, :]

# --- [1. Early Split] ---
total_len = len(log_returns)
train_size = int(0.7 * total_len)

train_raw = df_log_returns.iloc[:train_size, :]
test_raw = df_log_returns.iloc[train_size:, :]
train_init_raw = init_price_raw[:train_size]
test_init_raw = init_price_raw[train_size:]

# --- [2. Scaling (6차원, ablation 의 w/o RV 와 동일: SS1 -> LambertW -> SS2)] ---
scaled6_full, scalers6 = scaling(df_log_returns.values)
train6 = scaled6_full[:train_size]

# --- [3. Windowing] ---  test_data 는 평가용 real 6차원
test_windowed = torch.from_numpy(rolling_window(test_raw.values, seq_len)).float()
test_data = test_windowed.clone().detach()  # (B, 256, 6)

print("🚀 Benchmark 데이터 준비 완료")
print(f"  train6 scaled: {train6.shape}, scalers: {len(scalers6)}")
print(f"  test_data (real 6d): {tuple(test_data.shape)}")

# Benchmark 공통 유틸
학습/검증/생성/평가/종합 함수 정의. (이 셀을 한 번 실행한 뒤 1→2→3 순서로 진행)

In [ ]:
# ===== Benchmark 공통 유틸 =====
# QuantGAN(풀) / TailGAN(코어 적응형) 을 사용자 6차원 데이터·scaling·평가 파이프라인으로 학습/생성/평가.
# 구조는 2_abligation_study 와 동일: 섹션별 train+validate -> 마지막에 generate+evaluate+종합표.
# (Preprocessing 셀 실행 후 사용: device, seq_len, train6, scalers6, train_raw, test_data,
#  train_init_raw, test_init_raw)

N_RUNS = 5
TARGET_N = 2000
CLIP_LATENT = 8.0          # 역변환 전 latent clip (LambertW overflow 방지)
NUM_EPOCHS_QG = 300        # QuantGAN epochs (quantGAN.ipynb 동일)
SAVE_EVERY_QG = 5
NUM_EPOCHS_TG = 300        # TailGAN epochs
SAVE_EVERY_TG = 5

BENCH = {}                 # 벤치마크 레지스트리 (섹션에서 채움; 섹션 3 에서 일괄 생성/평가)


class SimpleConfig:
    def __init__(self, **kw):
        self.__dict__.update(kw)


def make_to_6d(scalers):
    """생성된 scaled 6차원(B,T,C) -> 원본 6차원 로그수익률(B,6,T)."""
    return lambda s, sc=scalers: inverse_scaling(s.transpose(1, 2), sc)


# ---------------- QuantGAN (full) ----------------
QG_NOISE, QG_HIDDEN = 6, 80   # noise_dim=6, hidden=80 (quantGAN.ipynb 동일)


def quantgan_build_G():
    cfg = SimpleConfig(noise_dim=QG_NOISE, G_hidden_dim=QG_HIDDEN, D_hidden_dim=QG_HIDDEN,
                       n_steps=seq_len, n_vars=6)
    return UserGenerator(cfg).to(device)


def quantgan_build_D():
    cfg = SimpleConfig(noise_dim=QG_NOISE, G_hidden_dim=QG_HIDDEN, D_hidden_dim=QG_HIDDEN,
                       n_steps=seq_len, n_vars=6)
    return UserDiscriminator(cfg).to(device)


@torch.no_grad()
def quantgan_sample(G, n):
    z = torch.randn(n, QG_NOISE, seq_len, device=device)
    return G(z).transpose(1, 2).cpu()   # (n, T, 6) scaled


def train_quantgan(tag, train_scaled, n_epochs=NUM_EPOCHS_QG, batch_size=256):
    # 원본 QuantGAN 의 GANTrainer 를 그대로 사용 -> 학습 루프 100% 동일
    from types import SimpleNamespace
    from src.baselines.trainer import GANTrainer
    print(f"\n{'#'*70}\n# TRAIN [{tag}] QuantGAN (원본 GANTrainer)\n{'#'*70}")
    training = torch.from_numpy(rolling_window(train_scaled, seq_len)).float()
    # 원본 quantGAN.ipynb cell3 과 동일: shuffle=False
    train_dl = DataLoader(TensorDataset(training), batch_size=batch_size, shuffle=False)
    config = SimpleNamespace(
        file_name="indices2", log_interval=5, n_vars=6, n_steps=seq_len,
        device=device, seed=0, batch_size=batch_size, n_epochs=n_epochs, algo="QuantGAN",
        G_hidden_dim=QG_HIDDEN, D_hidden_dim=QG_HIDDEN, output_dim=1, noise_dim=QG_NOISE,
        clip=0.01, lr_G=2e-4, lr_D=1e-4, D_steps_per_G_step=1,
    )
    G = UserGenerator(config)
    D = UserDiscriminator(config)
    count_parameters(G)
    clean_memory(device)

    wdir = file_manager.get_weight_path()
    os.makedirs(wdir, exist_ok=True)
    # GANTrainer.__init__ 이 './results/models/...' 를 하드코딩으로 만들기 때문에,
    # 생성자 동안만 os.makedirs 를 차단해 results/ 자체가 생기지 않게 한다.
    _orig_makedirs = os.makedirs
    def _skip_results(path, *a, **k):
        p = str(path).replace("\\", "/")
        if p.startswith("results/") or p.startswith("./results"):
            return
        return _orig_makedirs(path, *a, **k)
    os.makedirs = _skip_results
    try:
        trainer = GANTrainer(G=G, D=D, train_dl=train_dl, config=config)
    finally:
        os.makedirs = _orig_makedirs
    # 저장 경로를 우리 weight 폴더(result/weight)로 직접 지정
    trainer.results_dir = wdir
    trainer.fit()   # result/weight/Generator_{e}.pt 로 저장 (epoch%5==0)
    # 검증/생성이 찾는 규칙({tag}_{e}.pt)에 맞게 같은 폴더에서 이름만 변경
    for src_pt in glob.glob(f"{wdir}/Generator_*.pt"):
        e = int(os.path.basename(src_pt).replace("Generator_", "").replace(".pt", ""))
        os.replace(src_pt, f"{wdir}/{tag}_{e}.pt")
    print(f"✅ [{tag}] 학습 완료 → {wdir}/{tag}_*.pt")
    return G


# ---------------- TailGAN (core 적응형) ----------------
TG_LATENT, TG_BATCH, TG_NPORT = 100, 256, 20
TG_ALPHAS, TG_W, TG_TEMP = [0.05], 10.0, 0.01


def neural_sort(s, tau):
    """TailGAN NeuralSort (util.py) 의 device-agnostic 재구현."""
    n = s.size(1)
    one = torch.ones((n, 1), device=s.device, dtype=s.dtype)
    A_s = torch.abs(s - s.permute(0, 2, 1))
    Bm = A_s @ (one @ one.t())
    sc = (n + 1 - 2 * (torch.arange(n, device=s.device, dtype=s.dtype) + 1))
    C = s @ sc.unsqueeze(0)
    return torch.softmax(((C - Bm).permute(0, 2, 1)) / tau, dim=-1)


def _g1q(v, W): return -W * v ** 2 / 2
def _g2q(e, a): return a * e
def _g2iq(e, a): return a * e ** 2 / 2


def S_quant(v, e, X, alpha, W=TG_W):
    """Elicitable VaR·ES tail score (TailGAN, quant 버전)."""
    if alpha < 0.5:
        rt = ((X <= v).float() - alpha) * (_g1q(v, W) - _g1q(X, W)) \
            + 1. / alpha * _g2q(e, alpha) * (X <= v).float() * (v - X) \
            + _g2q(e, alpha) * (e - v) - _g2iq(e, alpha)
    else:
        ai = 1 - alpha
        rt = ((X >= v).float() - ai) * (_g1q(X, W) - _g1q(v, W)) \
            + 1. / ai * _g2q(-e, ai) * (X >= v).float() * (X - v) \
            + _g2q(-e, ai) * (v - e) - _g2iq(-e, ai)
    return torch.mean(rt)


def tail_score(val, pnl_real, alphas=TG_ALPHAS, W=TG_W):
    X = pnl_real.T
    loss = 0.0
    for i, a in enumerate(alphas):
        loss = loss + S_quant(val[:, [2 * i]], val[:, [2 * i + 1]], X, a, W)
    return loss


class TailGenerator(nn.Module):
    """TailGAN MLP generator: latent -> (B, n_vars, seq_len). (scaled 공간이라 [-1,1] clamp 제거)"""
    def __init__(self, latent_dim, n_vars, seq_len_):
        super().__init__()
        def blk(i, o, norm=True):
            l = [nn.Linear(i, o)]
            if norm:
                l.append(nn.BatchNorm1d(o, 0.8))
            l.append(nn.LeakyReLU(0.2, inplace=True))
            return l
        self.n_vars, self.seq_len = n_vars, seq_len_
        self.model = nn.Sequential(*blk(latent_dim, 128, False), *blk(128, 256),
                                   *blk(256, 512), *blk(512, 1024),
                                   nn.Linear(1024, n_vars * seq_len_))

    def forward(self, z):
        x = self.model(z)
        return x.view(x.shape[0], self.n_vars, self.seq_len)


class TailDiscriminator(nn.Module):
    """포트폴리오 terminal PnL 의 VaR·ES 를 추정 (NeuralSort + project)."""
    def __init__(self, batch_size, weights, alphas=TG_ALPHAS, W=TG_W, temp=TG_TEMP, project=True):
        super().__init__()
        self.register_buffer("weights", torch.as_tensor(weights, dtype=torch.float32))  # (n_port, n_vars)
        self.alphas, self.W, self.temp, self.project = alphas, W, temp, project
        self.model = nn.Sequential(nn.Linear(batch_size, 256), nn.LeakyReLU(0.2, inplace=True),
                                   nn.Linear(256, 128), nn.LeakyReLU(0.2, inplace=True),
                                   nn.Linear(128, 2 * len(alphas)))

    def project_op(self, val):
        for i, a in enumerate(self.alphas):
            v = val[:, 2 * i].clone(); e = val[:, 2 * i + 1].clone()
            ind = torch.sign(torch.as_tensor(0.5 - a))
            val[:, 2 * i] = ind * ((self.W * v < e).float() * v + (self.W * v >= e).float() * (v + self.W * e) / (1 + self.W ** 2))
            val[:, 2 * i + 1] = ind * ((self.W * v < e).float() * e + (self.W * v >= e).float() * self.W * (v + self.W * e) / (1 + self.W ** 2))
        return val

    def forward(self, R):                     # R: (B, n_vars, T)
        pnl = R.sum(dim=2) @ self.weights.T    # (B, n_port) terminal 누적 포트폴리오 수익
        pnl_t = pnl.T                          # (n_port, B)
        pnl_s = pnl_t.reshape(*pnl_t.shape, 1)
        perm = neural_sort(pnl_s, self.temp)
        val = self.model(torch.bmm(perm, pnl_s).reshape(*pnl_t.shape))
        if self.project:
            val = self.project_op(val)
        return pnl, val


def _tail_weights(n_vars, n_port):
    rng = np.random.default_rng(0)
    w_eq = np.ones((1, n_vars)) / n_vars
    w_rand = rng.standard_normal((n_port - 1, n_vars))
    w_rand /= np.abs(w_rand).sum(1, keepdims=True)
    return np.concatenate([w_eq, w_rand], 0)


def tailgan_build_G():
    return TailGenerator(TG_LATENT, 6, seq_len).to(device)


@torch.no_grad()
def tailgan_sample(G, n):
    z = torch.randn(n, TG_LATENT, device=device)
    return G(z).transpose(1, 2).cpu()   # (n, T, 6) scaled


def train_tailgan(tag, train_scaled, n_epochs=NUM_EPOCHS_TG, batch_size=TG_BATCH,
                  lr_G=1e-4, lr_D=1e-5, save_every=SAVE_EVERY_TG):
    print(f"\n{'#'*70}\n# TRAIN [{tag}] TailGAN-core (MLP G/D + VaR·ES tail-score)\n{'#'*70}")
    training = torch.from_numpy(rolling_window(train_scaled, seq_len)).float().transpose(1, 2)  # (B, C, T)
    dl = DataLoader(TensorDataset(training), batch_size=batch_size, shuffle=True, drop_last=True)
    weights = _tail_weights(6, TG_NPORT)
    G = tailgan_build_G()
    D = TailDiscriminator(batch_size, weights).to(device)
    optG = torch.optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))
    wdir = file_manager.get_weight_path()
    os.makedirs(wdir, exist_ok=True)
    count_parameters(G)
    clean_memory(device)
    G.train(); D.train()
    pbar = tqdm(range(1, n_epochs + 1))
    for epoch in pbar:
        dsum, gsum, nb = 0.0, 0.0, 0
        for (R,) in dl:
            R = R.to(device)
            z = torch.randn(R.shape[0], TG_LATENT, device=device)
            # --- D: real_score - fake_score ---
            optD.zero_grad()
            pnl_real, val_real = D(R)
            _, val_fake = D(G(z).detach())
            lossD = tail_score(val_real, pnl_real.detach()) - tail_score(val_fake, pnl_real.detach())
            lossD.backward(); optD.step()
            # --- G: fake_score (real pnl 분포를 맞추도록) ---
            optG.zero_grad()
            with torch.no_grad():
                pnl_x, _ = D(R)
            _, gval = D(G(z))
            lossG = tail_score(gval, pnl_x)
            lossG.backward(); optG.step()
            dsum += lossD.item(); gsum += lossG.item(); nb += 1
        pbar.set_description(f"[{tag}] E{epoch}")
        pbar.set_postfix(D=f"{dsum/max(nb,1):.3f}", G=f"{gsum/max(nb,1):.3f}")
        if epoch % save_every == 0 or epoch == n_epochs:
            torch.save(G.state_dict(), f"{wdir}/{tag}_{epoch}.pt")
    return G


# ---------------- 공통: 검증(best epoch) / 생성 / 평가 / 종합 ----------------
def validate_benchmark(tag, n_epochs, save_every, sample_n=100, emd_window=100, val_start=50):
    cfg = BENCH[tag]
    wdir = file_manager.get_weight_path()
    rp = file_manager.get_result_path(tag)
    os.makedirs(rp, exist_ok=True)
    G = cfg["build_G"]()

    # real 분포는 epoch 마다 고정 -> 롤링 분포를 '1번만' 전처리 (compute_avg_emd 의 real 재계산 제거)
    real_val = (
        torch.from_numpy(rolling_window(train_raw, seq_len)).float().transpose(1, 2).numpy()
    )
    real_dists = [
        rolling_window(real_val[:, i, :].T, emd_window).sum(axis=1).ravel()
        for i in range(real_val.shape[1])
    ]

    def fast_emd(f6):
        e = 0.0
        for i in range(f6.shape[1]):
            fd = rolling_window(f6[:, i, :].T, emd_window).sum(axis=1).ravel()
            e += wasserstein_distance(real_dists[i], fd)
        return e

    # 원본 quantGAN.ipynb cell18 처럼 val_start(기본 50) epoch 부터 save_every 간격 탐색
    epochs = list(range(val_start, n_epochs, save_every)) or list(range(save_every, n_epochs + 1, save_every))
    min_d, best = float("inf"), -1
    clean_memory(device)
    pbar = tqdm(epochs, desc=f"[{tag}] validate")
    for epoch in pbar:
        ck = f"{wdir}/{tag}_{epoch}.pt"
        if not os.path.exists(ck):
            continue
        G.load_state_dict(torch.load(ck, map_location=device))
        G.eval()
        emd = float("nan")
        with torch.no_grad():
            s = torch.clamp(cfg["sample"](G, sample_n), -CLIP_LATENT, CLIP_LATENT)
            with np.errstate(over="ignore", invalid="ignore"):
                f6 = cfg["to_6d_fn"](s)
            if np.isfinite(f6).all():
                emd = fast_emd(f6)
                if emd < min_d:
                    min_d, best = emd, epoch
        pbar.set_postfix(epoch=epoch, emd=f"{emd:.5f}", best=f"{min_d:.5f}@E{best}")
    clean_memory(device)

    if best == -1:
        print(f"❌ [{tag}] 유효한 체크포인트 없음")
        return -1
    print(f"🏆 [{tag}] Best Epoch: {best} (EMD: {min_d:.5f})")
    best_ck = f"{wdir}/{tag}_{best}.pt"
    G.load_state_dict(torch.load(best_ck, map_location=device))
    torch.save(G.state_dict(), f"{rp}/generator_best({best}).pt")
    for fp in glob.glob(f"{wdir}/{tag}_*.pt"):
        if fp != best_ck:
            try:
                os.remove(fp)
            except Exception:
                pass
    print(f"💾 [{tag}] Best generator 저장: {rp}/generator_best({best}).pt")
    return best


def generate_benchmark(tag, n_runs=N_RUNS, target_n=TA RGET_N):
    # 원본 quantGAN.ipynb 와 동일: target_n 개를 한 번에 생성 -> inverse_scaling 1번 (경계필터/재생성 없음)
    cfg = BENCH[tag]
    rp = file_manager.get_result_path(tag)
    best_files = glob.glob(f"{rp}/generator_best*.pt")
    if not best_files:
        raise FileNotFoundError(f"[{tag}] best generator 없음 ({rp}) - 먼저 학습/검증 실행")
    G = cfg["build_G"]()
    target_file = max(best_files, key=os.path.getmtime)
    G.load_state_dict(torch.load(target_file, map_location=device))
    G.eval()
    print(f"✅ [{tag}] 로드: {os.path.basename(target_file)}")

    real_data = test_data.transpose(1, 2).cpu().detach().numpy()
    for run_idx in range(n_runs):
        # CLIP_LATENT clamp 은 LambertW exp overflow(->inf, ValueError) 방지용 수치 안전장치일 뿐
        # (샘플을 버리는 필터가 아님). 그 외에는 원본과 동일하게 그대로 저장.
        s = torch.clamp(cfg["sample"](G, target_n), -CLIP_LATENT, CLIP_LATENT)
        with np.errstate(over="ignore", invalid="ignore"):
            fake = cfg["to_6d_fn"](s)
        np.save(os.path.join(rp, f"{run_idx}.npy"), fake)
        print(f"✅ [{tag}] Run {run_idx + 1}/{n_runs}: {fake.shape}")
    clean_memory(device)
    return real_data


def evaluate_benchmark(tag, real_data, n_runs=N_RUNS):
    """gen quality + portfolio 평가 (ablation 의 evaluate 와 동일), 파일명만 tag."""
    rp = file_manager.get_result_path(tag)
    windows = [1, 5, 20, 100]

    total_result_df = pd.DataFrame()
    for seed in range(n_runs):
        fake_data = np.load(f"{rp}/{seed}.npy")
        result_df = pd.DataFrame()
        result_df = pd.concat([result_df, calculate_ccf(real_data, fake_data, 0)], axis=0)
        total_CCF_df = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            total_CCF_df = pd.concat([total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1)
        result_df = pd.concat([result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=["Average"])], axis=0)
        result_df = pd.concat([result_df, calculate_distribution_scores(real_data, fake_data, windows)], axis=0)
        result_df = pd.concat([result_df, calculate_skew_kurtosis(real_data, fake_data, windows)], axis=0)
        _, acf_df = calculate_acf_score(real_data, fake_data, 40)
        result_df = pd.concat([result_df, acf_df], axis=0)
        _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
        result_df = pd.concat([result_df, leverage_df], axis=0)
        total_result_df = pd.concat([total_result_df, result_df], axis=1)
    statistics_df = pd.DataFrame(index=total_result_df.index, columns=["MEAN", "STD"])
    statistics_df["MEAN"] = total_result_df.mean(axis=1).values
    statistics_df["STD"] = total_result_df.std(axis=1).values
    statistics_df.to_csv(file_manager.get_result_file(f"{tag}_gen_quality"), encoding="utf-8-sig")

    config_dir = f"{LOCAL_BASE_DIRECTORY}/utils/eval_portfolio/config.yaml"
    with open(config_dir) as file:
        eval_config = ml_collections.ConfigDict(yaml.safe_load(file))
    alphas = eval_config.alphas
    strat_names = ["equal_weight", "mean_reversion", "trend_following", "vol_trading"]
    strat_map = {"equal_weight": "EW", "mean_reversion": "MR", "trend_following": "TF", "vol_trading": "VT"}
    eval_size = real_data.shape[0]
    for seed in range(n_runs):
        fake_data = np.load(f"{rp}/{seed}.npy")
        fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)
        real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)
        train_init_price = train_init_raw.reshape(-1, 1, 6).copy(); train_init_price[:] = 100
        test_init_price = test_init_raw.reshape(-1, 1, 6).copy(); test_init_price[:] = 100
        actual = fake_data_torch[:eval_size]
        n_samples = actual.shape[0]
        fake_prices = log_return_to_price(actual, torch.tensor(train_init_price[:n_samples], dtype=torch.float32, device=actual.device))
        real_prices = log_return_to_price(real_data_torch[:n_samples], torch.tensor(test_init_price[:n_samples], dtype=torch.float32, device=actual.device))
        if not (fake_prices > 0).all():
            raise ValueError("Sanity Check Failed: Some fake prices are not positive.")
        final_results = {}
        with torch.no_grad():
            for alpha in alphas:
                eval_config.Evaluation.TestMetrics.var.alpha = alpha
                eval_config.Evaluation.TestMetrics.es.alpha = alpha
                var_row, es_row = {}, {}
                for strat_name in strat_names:
                    subres_dict, _, _ = full_evaluation(fake_prices, real_prices, eval_config, strat_name=strat_name)
                    sn = strat_map[strat_name]
                    var_row[sn] = round(subres_dict.get("var_rel_mean", 0.0), 4)
                    es_row[sn] = round(subres_dict.get("es_rel_mean", 0.0), 4)
                var_row["Average"] = round(np.mean(list(var_row.values())), 4)
                es_row["Average"] = round(np.mean(list(es_row.values())), 4)
                final_results[f"RE_VaR({alpha})"] = var_row
                final_results[f"RE_ES({alpha})"] = es_row
        ordered_keys = [f"{m}({a})" for m in ["RE_VaR", "RE_ES"] for a in alphas]
        df_final = pd.DataFrame.from_dict(final_results, orient="index")
        cols_order = [strat_map[s] for s in strat_names] + ["Average"]
        df_final = df_final.reindex(index=ordered_keys, columns=cols_order)
        df_final.index.name = "Metric"
        df_final.T.to_csv(file_manager.get_result_file(f"{tag}_{seed}_portfolio"), encoding="utf-8-sig")
    print(f"✅ [{tag}] 개별 결과 저장: {tag}_gen_quality.csv, {tag}_<seed>_portfolio.csv")


def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""


def aggregate_gen_quality(tag_label_map, out_name="benchmark_summary_gen_quality"):
    cols = {}
    for tag, label in tag_label_map.items():
        f = file_manager.get_result_file(f"{tag}_gen_quality")
        if not os.path.exists(f):
            print(f"  (skip {tag}: {os.path.basename(f)} 없음)")
            continue
        df = pd.read_csv(f, index_col=0)
        cols[label] = pd.Series([_pm(m, s) for m, s in zip(df["MEAN"], df["STD"])], index=df.index)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


def aggregate_portfolio(tag_label_map, out_name="benchmark_summary_portfolio", n_runs=N_RUNS):
    cols = {}
    for tag, label in tag_label_map.items():
        parts = []
        for seed in range(n_runs):
            f = file_manager.get_result_file(f"{tag}_{seed}_portfolio")
            if os.path.exists(f):
                parts.append(pd.read_csv(f, index_col=0))
        if not parts:
            print(f"  (skip {tag}: portfolio csv 없음)")
            continue
        stacked = pd.concat(parts, axis=0)
        avg_rows = stacked.loc[["Average"]]
        m, sd = avg_rows.mean(axis=0), avg_rows.std(axis=0)
        cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, sd)], index=m.index)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


def build_bench_registry():
    """커널 재시작 후 '학습 없이' BENCH 재구성 (저장된 generator weight 로 생성/평가용)."""
    reg = {}
    reg["quantgan"] = dict(label="QuantGAN", build_G=quantgan_build_G,
                           sample=quantgan_sample, to_6d_fn=make_to_6d(scalers6))
    reg["tailgan"] = dict(label="TailGAN", build_G=tailgan_build_G,
                          sample=tailgan_sample, to_6d_fn=make_to_6d(scalers6))
    return reg

## 1. QuantGAN
`modules/QuantGAN` 의 `UserGenerator`/`UserDiscriminator`(TCN-GAN)를 그대로 import 해 학습.

In [8]:
# QuantGAN: 사용자 6차원 scaled 데이터로 TCN-GAN 학습 후 best epoch 선정
BENCH["quantgan"] = dict(
    label="QuantGAN",
    build_G=quantgan_build_G,
    sample=quantgan_sample,
    to_6d_fn=make_to_6d(scalers6),
)

G_qg = train_quantgan("quantgan", train6, n_epochs=NUM_EPOCHS_QG)
validate_benchmark("quantgan", NUM_EPOCHS_QG, SAVE_EVERY_QG)

Batch 5: D Loss: 0.8644026517868042, Generator Loss: 1.0414222478866577
Batch 10: D Loss: 1.0712175369262695, Generator Loss: 1.0067675113677979


 85%|████████▌ | 255/300 [14:47<02:36,  3.47s/it]

Batch 0: D Loss: 1.2210025787353516, Generator Loss: 1.0281696319580078
Batch 5: D Loss: 0.9097901582717896, Generator Loss: 1.0363917350769043
Batch 10: D Loss: 0.9320755004882812, Generator Loss: 0.9469977021217346


 85%|████████▌ | 256/300 [14:51<02:33,  3.49s/it]

Batch 0: D Loss: 1.0271782875061035, Generator Loss: 0.9843923449516296
Batch 5: D Loss: 0.9470601081848145, Generator Loss: 0.9926957488059998
Batch 10: D Loss: 0.9061399102210999, Generator Loss: 1.0181227922439575


...

100%|██████████| 300/300 [17:24<00:00,  3.48s/it]


✅ [quantgan] 학습 완료 → c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/weight/quantgan_*.pt


[quantgan] validate: 100%|██████████| 50/50 [01:05<00:00,  1.32s/it, best=0.26723@E280, emd=0.40745, epoch=295]

🏆 [quantgan] Best Epoch: 280 (EMD: 0.26723)
💾 [quantgan] Best generator 저장: c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/statistics/synthetic_data/quantgan/generator_best(280).pt


280

## 2. TailGAN (core)
TailGAN 핵심(포트폴리오 terminal PnL 의 VaR·ES tail-score + NeuralSort)만 6자산용으로 적응. `your_path`/argparse/디스크CSV 결합 제거.

In [9]:
# TailGAN(core): MLP G/D + VaR·ES tail-score 로 학습 후 best epoch 선정
BENCH["tailgan"] = dict(
    label="TailGAN",
    build_G=tailgan_build_G,
    sample=tailgan_sample,
    to_6d_fn=make_to_6d(scalers6),
)

G_tg = train_tailgan("tailgan", train6, n_epochs=NUM_EPOCHS_TG)
validate_benchmark("tailgan", NUM_EPOCHS_TG, SAVE_EVERY_TG)


######################################################################
# TRAIN [tailgan] TailGAN-core (MLP G/D + VaR·ES tail-score)
######################################################################
✅ 전체 파라미터 수: 2,280,832
✅ 학습 가능한 파라미터 수: 2,280,832


[tailgan] validate: 100%|██████████| 50/50 [01:01<00:00,  1.23s/it, best=3.32201@E50, emd=4.76325, epoch=295] 

🏆 [tailgan] Best Epoch: 50 (EMD: 3.32201)
💾 [tailgan] Best generator 저장: c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/statistics/synthetic_data/tailgan/generator_best(50).pt


50

## 3. 개별 결과 + Benchmark 종합 테이블
Best generator 로 합성데이터·개별 평가 CSV 생성 후, diffusion 포함 종합 테이블(`benchmark_summary_*.csv`, `±` 는 utf-8-sig)로 정리. 3-2 는 저장된 CSV 만으로 단독 실행 가능.

In [10]:
# ===== 3-1. 개별 결과: Best generator 로 합성데이터 생성 + 개별 평가 =====
# BENCH 가 비어 있으면(커널 재시작 등) 저장된 generator weight 기반으로 '학습 없이' 재구성.
# (Preprocessing 셀은 먼저 실행돼 있어야 함: scalers6, test_data, train_raw, ...)
if not BENCH:
    print("BENCH 가 비어 있어 build_bench_registry() 로 재구성합니다 (학습 생략, 저장된 weight 사용).")
    BENCH = build_bench_registry()
    for _t, _c in BENCH.items():
        _bf = glob.glob(f"{file_manager.get_result_path(_t)}/generator_best*.pt")
        print(f"  - {_t:10s} weight={'O ' + os.path.basename(_bf[0]) if _bf else 'X (학습 필요)'}")

for tag, cfg in BENCH.items():
    print(f"\n{'='*70}\n개별 결과 생성/평가: {cfg['label']} ({tag})\n{'='*70}")
    real_data = generate_benchmark(tag)
    evaluate_benchmark(tag, real_data)


개별 결과 생성/평가: QuantGAN (quantgan)
✅ [quantgan] 로드: generator_best(280).pt
✅ [quantgan] Run 1/5: (2000, 6, 256)
✅ [quantgan] Run 2/5: (2000, 6, 256)
✅ [quantgan] Run 3/5: (2000, 6, 256)
✅ [quantgan] Run 4/5: (2000, 6, 256)
✅ [quantgan] Run 5/5: (2000, 6, 256)
49
57
61
79
77
93
49
57
61
79
77
93
49
57
61
79
77
93
49
50
61
73
77
91
49
50
61
73
77
91
49
50
61
73
77
91
49
43
61
83
77
95
49
43
61
83
77
95
49
43
61
83
77
95
49
45
61
71
77
84
49
45
61
71
77
84
49
45
61
71
77
84
49
46
61
59
77
90
49
46
61
59
77
90
49
46
61
59
77
90
✅ [quantgan] 개별 결과 저장: quantgan_gen_quality.csv, quantgan_<seed>_portfolio.csv

개별 결과 생성/평가: TailGAN (tailgan)
✅ [tailgan] 로드: generator_best(50).pt
✅ [tailgan] Run 1/5: (2000, 6, 256)
✅ [tailgan] Run 2/5: (2000, 6, 256)
✅ [tailgan] Run 3/5: (2000, 6, 256)
✅ [tailgan] Run 4/5: (2000, 6, 256)
✅ [tailgan] Run 5/5: (2000, 6, 256)


ValueError: Sanity Check Failed: Some fake prices are not positive.

In [ ]:
# ===== 3-2. 종합 테이블 — 저장된 개별 결과 CSV 만으로 동작 (재생성/재학습 불필요) =====
# 고정 매핑이라 커널 재시작 후 '이 셀만' 단독 실행해도 디스크의 CSV 를 모아 종합표를 만든다.
# diffusion(ours, student_t) 결과가 있으면 함께 비교 (없으면 자동 skip).
tag_label_map = {
    "student_t": "Diffusion (ours)",
    "quantgan": "QuantGAN",
    "tailgan": "TailGAN",
}

print("결과 존재 여부 (X 면 해당 모델은 3-1 생성/평가 필요):")
for tag, label in tag_label_map.items():
    gq = os.path.exists(file_manager.get_result_file(f"{tag}_gen_quality"))
    pf = os.path.exists(file_manager.get_result_file(f"{tag}_0_portfolio"))
    print(f"  - {label:18s}: gen_quality={'O' if gq else 'X'}  portfolio={'O' if pf else 'X'}")

print("\n[Benchmark 종합 - Generation Quality]")
summary_gen_quality = aggregate_gen_quality(tag_label_map)
print(summary_gen_quality)

print("\n[Benchmark 종합 - Portfolio (Average 전략)]")
summary_portfolio = aggregate_portfolio(tag_label_map)
print(summary_portfolio)